# Fan on versus fan off: gas response time

The M1 milestone measurement. The same small gas exposure is repeated with the fan on
and off, and the time for the MQ-2 Rs/R0 ratio to cover 10, 50 and 90 % of its change is
compared. A fan that shortens t90 is buying detection time; the fan_state ledger event
exists because of this number.

**Procedure.** After warm-up and a baseline, start a capture. For each run: set the fan
(`!fan on` or `!fan off` in the capture window), wait a minute, type `gas in` as a fixed
dose is released at a fixed distance (for example one two-second butane-lighter puff,
unlit, 30 cm from the sensor), type `gas out` after two minutes, then wait for the ratio
to recover before the next run. Alternate on and off, at least five runs each, same
distance, same dose, doors and windows closed. Do this in a ventilated space with nothing
lit nearby.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # run from tools/calibration, next to fxcal.py
import fxcal
import matplotlib.pyplot as plt

In [ ]:
CAPTURE = None  # e.g. "../../captures/fan-response-2026-10-02.csv"
EVENTS = None   # the matching .events.jsonl
COLUMN = "mq2_ratio"

if CAPTURE:
    cap, events = fxcal.load_capture(CAPTURE), fxcal.load_events(EVENTS)
else:
    cap, events = fxcal.demo_fan_runs()
if not CAPTURE:
    print("SYNTHETIC DATA. Set CAPTURE to a bench recording before drawing any conclusion.")

In [ ]:
ts, vs = fxcal.series(cap, COLUMN)
runs = {"on": [], "off": []}
for t_in, t_out in fxcal.mark_pairs(events):
    state = fxcal.fan_state_at(events, t_in)
    response = fxcal.response_times(ts, vs, t_in, t_out)
    runs.setdefault(state or "unknown", []).append(response)
    fmt = lambda v: "not reached" if v is None else f"{v:.1f} s"
    print(f"run at {t_in:7.0f} s  fan {state}:  t10 {fmt(response.t10)}  t50 {fmt(response.t50)}  t90 {fmt(response.t90)}")

summary = fxcal.compare_fan(runs["on"], runs["off"])
print()
for key, value in summary.items():
    print(f"  {key:>13}: {value:.2f}" if isinstance(value, float) else f"  {key:>13}: {value}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = {"on": "tab:blue", "off": "tab:orange"}
for state, responses in runs.items():
    for response in responses:
        window = [(t - response.t_start, v) for t, v in zip(ts, vs)
                  if response.t_start - 20 <= t <= response.t_start + 150]
        ax.plot([w[0] for w in window], [w[1] for w in window], color=colors.get(state, "grey"), alpha=0.7,
                label=f"fan {state}")
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys())
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("seconds after gas in")
ax.set_ylabel(COLUMN)
plt.show()

Report the medians and the run count, not a single best run. If the on and off
distributions overlap, say so: that is a result too.